In [22]:
import pandas as pd
import numpy as np
import requests
import os
import sqlite3
from datetime import datetime, timezone

In [30]:
conn = sqlite3.connect("screener.db")
query = """
SELECT
    p.pair_address,
    p.symbol,
    p.dex_id,
    p.chain_id,
    p.market_cap,
    p.fdv,
    p.pair_created_at,
    latest.price_usd        AS current_price,
    previous.price_usd      AS previous_price,
    latest.liquidity_usd    AS current_liquidity,
    previous.liquidity_usd  AS previous_liquidity,
    latest.volume_m5        AS current_volume,
    previous.volume_m5      AS previous_volume,
    latest.snapshot_at
FROM pools p
JOIN pool_snapshots latest
ON latest.id = (
            SELECT id
            FROM pool_snapshots
            WHERE pair_address = p.pair_address
            ORDER BY snapshot_at DESC
            LIMIT 1
)
JOIN pool_snapshots previous
ON previous.id = (
            SELECT id
            FROM pool_snapshots
            WHERE pair_address = p.pair_address
            ORDER BY snapshot_at DESC
            LIMIT 1 OFFSET 1
)
"""

df = pd.read_sql_query(query, conn)
conn.close()
df["pair_created_at"] = pd.to_datetime(df["pair_created_at"], unit="ms",utc=True)
df["price_change_pct"] = ((df["current_price"] - df["previous_price"]) / df["previous_price"]) * 100
df["liquidity_change_pct"] = ((df["current_liquidity"] - df["previous_liquidity"]) / df["previous_liquidity"]) * 100
df["volume_change_pct"] = ((df["current_volume"] - df["previous_volume"]) / df["previous_volume"]) * 100
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(df.head())


                                   pair_address symbol       dex_id chain_id  \
0    0xE520a7C2d2Ed54FA9d50Cf2BAf3969148BbdF46b    E3D  pancakeswap      bsc   
1  C1KHGdzEh3DVr7iE8tUiqUuox9eYigLFBhikpU3rbjwD  ANSOM     pumpswap   solana   
2  3ipHnTpz72RGsQgGPJ54PNqdN2Svne98akGZAgUye3hw  ANSOM      meteora   solana   
3  BajPdk4gBPtBohUQNofoYKLfpdEi9yRRHpioEzBeDXBZ  ANSOM      meteora   solana   
4  D216BgduYfZodoyPPLcN2tqY78YcD2GfhbyDXVufj9XY  ANSOM      meteora   solana   

   market_cap       fdv           pair_created_at  current_price  \
0        43.0      43.0 2026-06-28 03:54:25+00:00       0.233800   
1     56418.0   56418.0 2026-06-28 12:29:46+00:00       0.000080   
2    106894.0  106894.0 2026-06-28 12:30:04+00:00       0.000041   
3     94110.0   94110.0 2026-06-28 12:30:21+00:00       0.000046   
4    110238.0  110238.0 2026-06-28 12:32:31+00:00       0.000101   

   previous_price  current_liquidity  previous_liquidity  current_volume  \
0        0.233800             110.

In [35]:
BOT_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN')
CHAT_ID = os.getenv('CHAT_ID')

def send_telegram(message):
    requests.post(
        f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
        data={
            "chat_id": CHAT_ID,
            "text": message
        }
    )

send_telegram("The bot is ready to go ✅")

In [ ]:
SIGNAL_TIERS = [
    (15, "💎 GEM"),
    (12, "🚀 EXPLOSIVE"),
    (9,  "🔥 STRONG"),
    (6,  "🟢 GOOD"),
    (3,  "🟡 WATCH"),
]

SCORE_RULES = {
    "price_change_pct": [
        (50,  3, "🚀 Explosive Price Move"),
        (25,  2, "🚀 Strong Price Breakout"),
        (10,  1, "📈 Price Rising"),
    ],
    "volume_change_pct": [
        (300, 3, "🔥 Massive Volume Surge"),
        (100, 2, "🔥 Heavy Volume"),
        (50,  1, "📈 Volume Increasing"),
    ],
    "current_liquidity": [
        (100_000, 3, "🏦 Deep Liquidity"),
        (50_000,  2, "🏦 Healthy Liquidity"),
        (20_000,  1, "💦 Good Liquidity"),
    ],
}


def signal_strength(score):
    for threshold, label in SIGNAL_TIERS:
        if score >= threshold:
            return label
    return None


def _apply_tier_rules(pool, score, reasons):
    for field, tiers in SCORE_RULES.items():
        value = pool[field]
        for threshold, points, label in tiers:
            if value >= threshold:
                score += points
                reasons.append(label)
                break 
    return score, reasons


def _score_liquidity_change(pool, score, reasons):
    liq_chg = pool["liquidity_change_pct"]
    if liq_chg >= 10:
        score += 2; reasons.append("💧 Liquidity Growing")
    elif liq_chg >= 5:
        score += 1; reasons.append("💧 Liquidity Stable")
    elif liq_chg <= -20:
        score -= 2; reasons.append("⚠️ Liquidity Leaving")
    return score, reasons


def _score_market_cap(pool, score, reasons):
    mc = pool["market_cap"]
    if pd.isna(mc):
        return score, reasons
    if mc >= 500_000:
        score += 2; reasons.append("🏛 Large Market Cap")
    elif mc >= 100_000:
        score += 1; reasons.append("🏛 Healthy Market Cap")
    elif mc < 20_000:
        score -= 2; reasons.append("⚠️ Tiny Market Cap")
    return score, reasons


def _score_fdv(pool, score, reasons):
    mc, fdv = pool["market_cap"], pool["fdv"]
    if pd.isna(fdv) or fdv <= 0 or pd.isna(mc):
        return score, reasons
    ratio = mc / fdv
    if ratio >= 0.9:
        score += 2; reasons.append("✅ Healthy FDV")
    elif ratio >= 0.7:
        score += 1; reasons.append("✅ Good FDV")
    return score, reasons


def _score_volume_to_liquidity(pool, score, reasons):
    liquidity = pool["current_liquidity"]
    if liquidity <= 0:
        return score, reasons
    ratio = pool["current_volume"] / liquidity
    if ratio >= 3:
        score += 2; reasons.append("🔥 Exceptional Trading Activity")
    elif ratio >= 1:
        score += 1; reasons.append("📈 Active Trading")
    return score, reasons


def _score_pool_age(pool, score, reasons):
    try:
        age_hours = (
            datetime.now(timezone.utc) - pool["pair_created_at"]
        ).total_seconds() / 3600
    except TypeError:
        return score, reasons

    if age_hours <= 6:
        score += 3; reasons.append("🆕 New Pool (<6h)")
    elif age_hours <= 24:
        score += 2; reasons.append("🆕 New Pool (<24h)")
    elif age_hours <= 72:
        score += 1; reasons.append("🆕 New Pool (<3d)")
    return score, reasons


def score_pool(pool):
    if pd.isna(pool["current_liquidity"]) or pool["current_liquidity"] < 5_000:
        return None, []

    score, reasons = 0, []

    score, reasons = _apply_tier_rules(pool, score, reasons)
    score, reasons = _score_liquidity_change(pool, score, reasons)
    score, reasons = _score_market_cap(pool, score, reasons)
    score, reasons = _score_fdv(pool, score, reasons)
    score, reasons = _score_volume_to_liquidity(pool, score, reasons)
    score, reasons = _score_pool_age(pool, score, reasons)

    if pool["current_volume"] < 1_000:
        score -= 2; reasons.append("⚠️ Low Volume")

    return score, reasons

In [ ]:
def _fmt_price(val):
    """Auto-selects decimal places based on magnitude."""
    if pd.isna(val):
        return "N/A"
    if val < 0.0001:
        return f"${val:.10f}"
    if val < 0.01:
        return f"${val:.6f}"
    return f"${val:.4f}"

def _fmt_usd(val):
    return f"${val:,.0f}" if pd.notna(val) else "N/A"

def _fmt_pct(val):
    return f"{val:+.2f}%" if pd.notna(val) else "N/A"

def _fmt_age(pair_created_at):
    try:
        age_hours = (
            datetime.now(timezone.utc) - pair_created_at
        ).total_seconds() / 3600
        if age_hours < 1:
            return f"{int(age_hours * 60)}m ago"
        if age_hours < 24:
            return f"{age_hours:.1f}h ago"
        return f"{age_hours / 24:.1f}d ago"
    except TypeError:
        return "Unknown"


def build_message(pool, score, reasons):
    strength = signal_strength(score)

    return f"""
{strength}

🪙 {pool['symbol']}
🏦 {pool['dex_id'].upper()}

━━━━━━━━━━━━━━

💵 Price       {_fmt_price(pool['current_price'])} ({_fmt_pct(pool['price_change_pct'])})
💧 Liquidity   {_fmt_usd(pool['current_liquidity'])} ({_fmt_pct(pool['liquidity_change_pct'])})
🔥 Volume      {_fmt_usd(pool['current_volume'])} ({_fmt_pct(pool['volume_change_pct'])})
🏛 Market Cap  {_fmt_usd(pool['market_cap'])}
📅 Created     {_fmt_age(pool['pair_created_at'])}

━━━━━━━━━━━━━━

Signals
{chr(10).join(reasons)}

━━━━━━━━━━━━━━

⭐ Score: {score}
🔗 https://dexscreener.com/{pool['chain_id']}/{pool['pair_address']}
""".strip()


MIN_SCORE = 6
def score_dataframe(df):
    """Score all pools, return sorted list of alert dicts."""
    results = []
    for _, pool in df.iterrows():
        score, reasons = score_pool(pool)
        if score is not None and score >= MIN_SCORE:
            results.append({"pool": pool, "score": score, "reasons": reasons})
    return sorted(results, key=lambda x: x["score"], reverse=True)


def dispatch_alerts(df):
    alerts = score_dataframe(df)
    print(f"Dispatching {len(alerts)} alerts...")
    for alert in alerts:
        message = build_message(alert["pool"], alert["score"], alert["reasons"])
        send_telegram(message)

dispatch_alerts(df)

Dispatching 7 alerts...
